## Image preparation

The below code can be used to transform the images in an input directory (Input_Dir) 
to the right size (e.g. 32x32 pixels) into an output directory (Output_Dir).

Note: Duplicates will be removed by evaluating the file hash

### Basic Parameter

IMPORTANT: Do not rename any variables in this section — they are externally referenced in the GitHub action `Train Model`.

* `Input_Dir`: Path to the input directory containing training images
* `Output_Dir`: Path to the output directory where results (models, logs, etc.) will be saved
* `Target_Shape`: Tuple specifying the image dimensions (width, height, channels)

In [ ]:
# Parameters
Input_Dir = 'data_raw_all'
Output_Dir = 'data_resize_all'

# Target image size [x, y, channels]
Target_Shape = (32, 32, 3)


### Load Libraries

In [ ]:
import glob
import os

from pathlib import Path
import hashlib

import tensorflow as tf


### Delete Output Directory

In [ ]:
files = glob.glob(Output_Dir + '/*.jpg')

# Delete files
for f in files:
    os.remove(f)

print(str(len(files)) + " files have been deleted.")


### Load Files and Resize

In [ ]:
files = glob.glob(Input_Dir + '/*.jpg')
hashes = {}

# Process files
for i, file in enumerate(files):
    # Read image file
    image_bytes = tf.io.read_file(file)
    image = tf.image.decode_image(image_bytes, channels=Target_Shape[2], expand_animations=False)

    # Compute SHA256 hash of image content only, not of file itself (exclude metadata, etc)
    hash_val = hashlib.sha256(tf.io.encode_jpeg(image).numpy()).hexdigest()
    if hash_val in hashes:
        hashes[hash_val].append(file)
        continue
    else:
        hashes[hash_val] = [file]
    
    # Resize image
    image = tf.image.resize(image, [Target_Shape[0], Target_Shape[1]], method=tf.image.ResizeMethod.MITCHELLCUBIC)
    image = tf.cast(image, tf.uint8)
    image = tf.io.encode_jpeg(image, quality=100)

    # Save resized image
    base = os.path.basename(file)
    save_path = os.path.join(Output_Dir, base)
    tf.io.write_file(save_path, image)

    # Log every 500th image
    if i % 500 == 0:
        print(f" {i} images processed...")

print(f"Completed. {i+1} images processed")


### Remove Duplicates

In [ ]:
# Duplicate files are a risk to the metrics, they pollute the validation dataset
for hash in hashes:
    if len(hashes[hash]) > 1:
        print(f"Duplicates: {hashes[hash]}")    
        for duplicate in hashes[hash][1:]:
            # Remove all except the first
            print(f"Removed: {duplicate}")
            os.remove(duplicate)
